# Practical 01 — Python for AI - Environment, Data Structures and the Toolkit

**Artificial Intelligence 3(2-1) · BS Computer Science · Semester III**

| | |
|---|---|
| **Week** | 1 |
| **Related lectures** | Lectures 1 and 2 |
| **Duration** | 3 hours |
| **Submission** | This notebook, run top to bottom from a clean kernel |

## 1. Objectives

By the end of this session you will be able to:

1. Set up and verify the Python environment used throughout this course.
2. Use the data structures that every later practical depends on: lists, dictionaries, sets and tuples.
3. Write and use generators, and explain how they differ from lists.
4. Represent an agent's percepts, actions and belief state in Python.
5. Implement the sense-update-decide-act loop from Lecture 2.

## 2. Background theory

Lecture 2 described an agent as running a loop: it senses the environment, updates a bounded summary of the past called the **belief state**, decides on an action from that summary alone, and acts. Today you build that loop in Python.

The programming ideas here are not incidental. Every later practical uses dictionaries to map situations to values, sets to record what has already been visited, and generators to produce candidate moves one at a time rather than building an enormous list in memory. If these are unfamiliar now, the search practicals in weeks 4 and 5 will be much harder than they need to be.

**Key points to keep in mind while you work:**

- A belief state is whatever the agent remembers; anything not in it cannot influence its behaviour.
- A generator produces values on demand, so it never builds the whole sequence in memory.
- A dictionary lookup is fast regardless of size, which is why we use them for state tables.
- A set membership test is also fast, which is why we use sets to record visited states.

## 3. Apparatus and setup

**Software:** Python 3.10 or later, Jupyter Notebook or JupyterLab, NumPy, Matplotlib.

Run the cell below first. If any import fails, install the missing package with `pip install <package>` and run the cell again before continuing.

In [1]:
! pip install numpy matplotlib 

  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 6.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 9.8 MB/s  0:00:00m eta 0:00:01
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 9.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 9.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [matplotlib]8 [matplotlib]

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys
print('Python',sys.version.split()[0])
import numpy, matplotlib
print('numpy', numpy.__version__)
print('matplotlib', matplotlib.__version__)
print('\nEnvironment ready.')

Python 3.14.6
numpy 2.5.3
matplotlib 3.11.2

Environment ready.


## 4. Procedure

Work through these steps in order. The code here is given to you and should run as it stands — read it, run it, and make sure you understand what each part does before moving on to your own tasks in Section 5.

### Step 1 — The data structures you will actually use

Run this and read the output carefully. Each of these four structures is used for a specific purpose in later practicals, noted in the comments.

In [3]:
# A list: an ordered sequence you can add to. Used for the frontier in search.
frontier = ['A', 'B', 'C']
frontier.append('D')
print('frontier      :', frontier)

# A dictionary: a lookup table. Used for mapping states to values.
distance = {'A': 0, 'B': 3, 'C': 7}
print('distance to B :', distance['B'])

# A set: fast membership testing. Used to record what we have already seen.
visited = {'A', 'B'}
print('seen C?       :', 'C' in visited)

# A tuple: a fixed group of values. Used for coordinates and immutable states.
position = (3, 5)
print('position      :', position, 'row', position[0])

frontier      : ['A', 'B', 'C', 'D']
distance to B : 3
seen C?       : False
position      : (3, 5) row 3


### Step 2 — Generators: producing values one at a time

A generator computes each value only when it is asked for. In search problems the list of possible next moves can be enormous, so producing them one at a time matters. Note what happens when you consume a generator twice.

In [ ]:
def squares_list(n):
    return [x * x for x in range(n)]        # builds the whole list at once

def squares_gen(n):
    for x in range(n):
        yield x * x                          # produces one value at a time

print('list   :', squares_list(5))
print('gen    :', list(squares_gen(5)))

g = squares_gen(5)
print('first sum :', sum(g))
print('second sum:', sum(g), '   <- a generator is consumed once')

list   : [0, 1, 4, 9, 16]
gen    : [0, 1, 4, 9, 16]
first sum : 30
second sum: 0    <- a generator is consumed once


### Step 3 — An agent and its environment

Here is a minimal agent in a corridor of rooms. It can move left or right, it senses only whether the current room is dirty, and its belief state records which rooms it has already cleaned. Read the loop and identify the four stages from Lecture 2.

In [ ]:
import random
random.seed(0)

class Environment:
    def __init__(self, n_rooms=5):
        self.n = n_rooms
        self.dirty = {r: random.choice([True, False]) for r in range(n_rooms)}
        self.position = 0

    def percept(self):
        # The agent sees ONLY whether the current room is dirty.
        return {'dirty': self.dirty[self.position]}

    def act(self, action):
        if action == 'clean':
            self.dirty[self.position] = False
        elif action == 'right' and self.position < self.n - 1:
            self.position += 1
        elif action == 'left' and self.position > 0:
            self.position -= 1

env = Environment()
print('true state (hidden from the agent):', env.dirty)
print('what the agent can see           :', env.percept())

true state (hidden from the agent): {0: False, 1: False, 2: True, 3: False, 4: False}
what the agent can see           : {'dirty': False}


### Step 4 — The sense-update-decide-act loop

This agent keeps a belief state: the set of rooms it believes it has cleaned. Notice that the decision uses only the belief state and the current percept.

In [ ]:
def run_agent(env, steps=12, verbose=True):
    cleaned = set()                       # THE BELIEF STATE
    for t in range(steps):
        percept = env.percept()           # SENSE
        if percept['dirty']:              # UPDATE + DECIDE
            action = 'clean'
            cleaned.add(env.position)
        else:
            action = 'right'
        if verbose:
            print('t=%2d  room=%d  percept=%-5s  action=%-5s  belief=%s'
                  % (t, env.position, percept['dirty'], action, sorted(cleaned)))
        env.act(action)                   # ACT
    return cleaned

env = Environment()
cleaned = run_agent(env)
print('\nrooms still dirty:', [r for r, d in env.dirty.items() if d])

t= 0  room=0  percept=True   action=clean  belief=[0]
t= 1  room=0  percept=False  action=right  belief=[0]
t= 2  room=1  percept=False  action=right  belief=[0]
t= 3  room=2  percept=True   action=clean  belief=[0, 2]
t= 4  room=2  percept=False  action=right  belief=[0, 2]
t= 5  room=3  percept=True   action=clean  belief=[0, 2, 3]
t= 6  room=3  percept=False  action=right  belief=[0, 2, 3]
t= 7  room=4  percept=False  action=right  belief=[0, 2, 3]
t= 8  room=4  percept=False  action=right  belief=[0, 2, 3]
t= 9  room=4  percept=False  action=right  belief=[0, 2, 3]
t=10  room=4  percept=False  action=right  belief=[0, 2, 3]
t=11  room=4  percept=False  action=right  belief=[0, 2, 3]

rooms still dirty: []


## 5. Tasks

These are your work. Each cell marked `# TODO` is for you to complete. Write your own code; do not copy the procedure cells unchanged.

### Task 1 — Predict before you run

Write a generator that produces the first n even numbers. Before running it, write in a comment what you expect `sum()` to return the second time it is called on the same generator, and then check.

> **Marks:** 2

In [ ]:
# TODO
# My prediction for the second sum: 
# sum = 2,4,6,8 =20 
def pass_even(n):
    for i in range(n):
        if i % 2 == 0:
            yield  i
# Yahan hum 5 even numbers ka generator bana rahe hain
evens = pass_even(10)
first_sum = sum(evens)
print("First sum:", first_sum)



First sum: 20


### Task 2 — Give the agent a better policy

The agent above always moves right, so it walks off the end of the corridor and stops making progress. Write a new policy that uses the belief state to turn around when it reaches the end. Run it and confirm every room is cleaned.

> **Marks:** 4

In [ ]:
#Look at the belief state to see if the robot is likely in the last coridoor on the right.
#If the chance of being at the right end is high, stop moving right and change direction to LEFT.
#Keep moving and cleaning until all rooms are clean.

### Task 3 — Break the belief state deliberately

Remove the belief state from your agent — that is, stop recording which rooms have been cleaned — and run it again. Describe in a comment what changes about its behaviour, and explain why in terms of Lecture 2.

> **Marks:** 4

In [ ]:
# As a result, it gets stuck in an infinite loop.moving back and forth.
# because it can never be certain that the job is finished or when to stop.

### Task 4 — Count the work

Modify your agent to count how many actions it takes to clean every room. Run it on corridors of 5, 10 and 20 rooms and record the counts in the observations table.

> **Marks:** 3

In [13]:
# TODO: (Formula: For an N-room corridor starting at one end, cleaning takes $N$ CLEAN actions + $N-1$ movement actions = 2N - 1 total actions).
def run_experiment(num_rooms):
    actions = 0
    pos = 0
    direction = 1  #
    rooms = [True] * num_rooms  
    while True in rooms:
        if rooms[pos]:
            rooms[pos] = False
            actions += 1

        if True not in rooms:
            break

        if pos == num_rooms - 1 and direction == 1:
            direction = -1
        elif pos == 0 and direction == -1:
            direction = 1

        # 3. Move
        pos += direction
        actions += 1

    return actions

# Run for 5, 10, and 20 rooms
print("| Rooms | Actions |")
print("|-------|---------|")
for size in [5, 10, 20]:
    count = run_experiment(size)
    print(f"| {size:<5} | {count:<7} |")





| Rooms | Actions |
|-------|---------|
| 5     | 9       |
| 10    | 19      |
| 20    | 39      |


### Task 5 — Describe your agent in four parts

In a markdown cell below, describe your agent using the four parts from Lecture 1: abilities, goals, prior knowledge, and experience. Be specific — name the actual variables in your code where you can.

> **Marks:** 2

In [ ]:
# TODO: write your four-part description in a markdown cell below this one:


#Abilities: Can move (LEFT, RIGHT) by modifying positions, and clean the current location by setting rooms[pos] = False.
#Goals: Clean all rooms using the minimum steps (actions).
#Prior Knowledge: Knows the corridor boundaries (pos == 0 and pos == num_rooms - 1) and turns around (direction = -1 or 1) to avoid walking off the edge.
#Experience: Reads current room status (rooms[pos]) at each step and updates its position (pos += direction) as it moves.

## 6. Observations

Complete this table from your own results. Do not leave it blank — the observations carry marks and are what the viva is based on.

| Corridor size | Actions taken (with belief state) | Actions taken (without) | Every room cleaned? |
|---|---|---|---|
| 5 rooms  |9 | unlimited |yes  |
| 10 rooms |19| unlimited|  yes|
| 20 rooms |20| unlimited |yes  |

## 7. Discussion

Answer in your own words, in the cell below. Two or three sentences each is enough.

1. What exactly does your agent's belief state contain, and what did you deliberately leave out of it?
2. Your agent senses only the current room. Give a situation where two different rooms produce identical percepts but require different actions.
3. How did the number of actions grow as the corridor got longer? Was the growth roughly proportional, or worse?

*Your answers:*

1. Belief State Contents: Tracks the agent's current position and which rooms are clean versus dirty. Deliberately left out are step counts, action history, and physical environmental details like room dimensions.
2. Different Actions: A dirty room at the far-right end and a dirty room in the middle both return the exact same percept (DIRT).After cleaning, the end room requires turning LEFT, while the middle room requires continuing RIGHT.
3. Growth Rate of Actions: The actions grew linearly and were directly proportional to corridor size, exactly following 2N - 1. Doubling the rooms simply doubled the steps needed.


## 8. Submission checklist

- [ ] Notebook runs top to bottom from a clean kernel with no errors
- [ ] Every `# TODO` cell is completed with your own code
- [ ] The observations table is filled in with your actual results
- [ ] Discussion answers are written in your own words
- [ ] Your name and roll number are in the cell below
- [ ] File named `P01_<your-roll-number>.ipynb`

In [ ]:
NAME = "MUHAMMAD HASHAIR"
ROLL_NUMBER = "CSE-2025-048"
print(NAME, ROLL_NUMBER)

MUHAMMAD HASHAIR CSE-2025-048
